# Part B - Q1: 15-class Classification with Pretrained Models

**Task:** Using pretrained `resnet18`, `densenet121` and `vgg19` models, train a
15-class classification model for **3 epochs** and report **per-class precision and
recall** on the test set.

**Setup / interpretation:**
* Dataset has 15 classes; for every class images `image_0001`..`image_0040` are used
  for **training** and the remaining images for **testing**.
* Here the pretrained convolutional backbone is used as a **fixed feature extractor**
  (backbone frozen) and only a new 15-way linear classifier head is trained. This keeps
  Q1 distinct from Q2, where the `vgg19` network is fully fine-tuned.

In [1]:
import os, random, numpy as np, torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.metrics import classification_report, precision_recall_fscore_support, confusion_matrix

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

DATA_ROOT = os.path.join("data", "classification", "dataset")
classes = sorted(d for d in os.listdir(DATA_ROOT) if os.path.isdir(os.path.join(DATA_ROOT, d)))
cls2idx = {c: i for i, c in enumerate(classes)}
print(f"{len(classes)} classes:", classes)

Using device: cuda
15 classes: ['accordion', 'bass', 'camera', 'crocodile', 'crocodile_head', 'cup', 'dollar_bill', 'emu', 'gramophone', 'hedgehog', 'nautilus', 'pizza', 'pyramid', 'sea_horse', 'windsor_chair']


## Build the train/test split
For every class the first 40 images (`image_0001`..`image_0040`) form the training set
and all remaining images form the test set.

In [2]:
def build_split(root):
    train_items, test_items = [], []
    for c in classes:
        files = sorted(os.listdir(os.path.join(root, c)))
        files = [f for f in files if f.lower().endswith((".jpg", ".jpeg", ".png"))]
        for f in files:
            # file names look like image_0001.jpg -> extract the integer index
            num = int("".join(ch for ch in os.path.splitext(f)[0] if ch.isdigit()))
            path = os.path.join(root, c, f)
            (train_items if num <= 40 else test_items).append((path, cls2idx[c]))
    return train_items, test_items

train_items, test_items = build_split(DATA_ROOT)
print(f"Train images: {len(train_items)}  |  Test images: {len(test_items)}")

Train images: 600  |  Test images: 205


In [3]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class ImageList(Dataset):
    def __init__(self, items, tf):
        self.items, self.tf = items, tf
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        path, label = self.items[i]
        img = Image.open(path).convert("RGB")
        return self.tf(img), label

train_ds = ImageList(train_items, train_tf)
test_ds = ImageList(test_items, eval_tf)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=0)

## Model factory (pretrained backbone frozen, new 15-way head)

In [4]:
def build_model(name, num_classes=15):
    if name == "resnet18":
        m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        for p in m.parameters():
            p.requires_grad = False
        m.fc = nn.Linear(m.fc.in_features, num_classes)
    elif name == "densenet121":
        m = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
        for p in m.parameters():
            p.requires_grad = False
        m.classifier = nn.Linear(m.classifier.in_features, num_classes)
    elif name == "vgg19":
        m = models.vgg19(weights=models.VGG19_Weights.IMAGENET1K_V1)
        for p in m.parameters():
            p.requires_grad = False
        in_f = m.classifier[6].in_features
        m.classifier[6] = nn.Linear(in_f, num_classes)
    else:
        raise ValueError(name)
    return m.to(device)

In [5]:
def train_and_eval(name, epochs=3, lr=1e-3):
    print(f"\n===== {name} =====")
    model = build_model(name)
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.Adam(params, lr=lr)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(1, epochs + 1):
        model.train()
        running, correct, total = 0.0, 0, 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            running += loss.item() * x.size(0)
            correct += (out.argmax(1) == y).sum().item()
            total += x.size(0)
        print(f"  epoch {epoch}/{epochs}  loss={running/total:.4f}  train_acc={correct/total:.4f}")

    # evaluation
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for x, y in test_loader:
            x = x.to(device)
            preds = model(x).argmax(1).cpu().numpy()
            y_pred.extend(preds.tolist())
            y_true.extend(y.numpy().tolist())
    print(f"\nPer-class precision / recall for {name} (test set):")
    print(classification_report(y_true, y_pred, target_names=classes, digits=3, zero_division=0))
    p, r, f, s = precision_recall_fscore_support(y_true, y_pred, labels=list(range(len(classes))), zero_division=0)
    return {"name": name, "precision": p, "recall": r, "f1": f, "support": s,
            "acc": float(np.mean(np.array(y_true) == np.array(y_pred)))}

In [6]:
results = {}
for name in ["resnet18", "densenet121", "vgg19"]:
    results[name] = train_and_eval(name)


===== resnet18 =====


  epoch 1/3  loss=2.2411  train_acc=0.3600


  epoch 2/3  loss=1.1245  train_acc=0.8367


  epoch 3/3  loss=0.6552  train_acc=0.9050



Per-class precision / recall for resnet18 (test set):
                precision    recall  f1-score   support

     accordion      0.938     1.000     0.968        15
          bass      0.824     1.000     0.903        14
        camera      1.000     1.000     1.000        10
     crocodile      0.778     0.700     0.737        10
crocodile_head      0.778     0.636     0.700        11
           cup      0.895     1.000     0.944        17
   dollar_bill      1.000     1.000     1.000        12
           emu      1.000     1.000     1.000        13
    gramophone      1.000     0.818     0.900        11
      hedgehog      1.000     0.929     0.963        14
      nautilus      1.000     0.933     0.966        15
         pizza      1.000     1.000     1.000        13
       pyramid      0.938     0.882     0.909        17
     sea_horse      0.833     0.882     0.857        17
 windsor_chair      0.941     1.000     0.970        16

      accuracy                          0.927  

...

100%|██████████| 548M/548M [00:13<00:00, 42.3MB/s]

  epoch 1/3  loss=1.0121  train_acc=0.7267


  epoch 2/3  loss=0.1677  train_acc=0.9500


  epoch 3/3  loss=0.1007  train_acc=0.9700



Per-class precision / recall for vgg19 (test set):
                precision    recall  f1-score   support

     accordion      1.000     1.000     1.000        15
          bass      1.000     1.000     1.000        14
        camera      1.000     1.000     1.000        10
     crocodile      0.571     0.800     0.667        10
crocodile_head      0.889     0.727     0.800        11
           cup      1.000     1.000     1.000        17
   dollar_bill      1.000     1.000     1.000        12
           emu      1.000     1.000     1.000        13
    gramophone      1.000     1.000     1.000        11
      hedgehog      1.000     0.929     0.963        14
      nautilus      0.938     1.000     0.968        15
         pizza      1.000     1.000     1.000        13
       pyramid      1.000     0.941     0.970        17
     sea_horse      0.933     0.824     0.875        17
 windsor_chair      0.941     1.000     0.970        16

      accuracy                          0.951     

## Summary: overall test accuracy of the three pretrained models

In [7]:
import pandas as pd
summary = pd.DataFrame({m: {"overall_accuracy": results[m]["acc"],
                            "mean_precision": float(np.mean(results[m]["precision"])),
                            "mean_recall": float(np.mean(results[m]["recall"]))}
                        for m in results}).T
print(summary.round(4))

             overall_accuracy  mean_precision  mean_recall
resnet18               0.9268          0.9282       0.9187
densenet121            0.9756          0.9683       0.9679
vgg19                  0.9512          0.9515       0.9480


In [8]:
# Per-class precision & recall table for each model
for m in results:
    df = pd.DataFrame({"precision": results[m]["precision"].round(3),
                       "recall": results[m]["recall"].round(3),
                       "support": results[m]["support"]}, index=classes)
    print(f"\n=== {m} per-class precision / recall ===")
    print(df)


=== resnet18 per-class precision / recall ===
                precision  recall  support
accordion           0.938   1.000       15
bass                0.824   1.000       14
camera              1.000   1.000       10
crocodile           0.778   0.700       10
crocodile_head      0.778   0.636       11
cup                 0.895   1.000       17
dollar_bill         1.000   1.000       12
emu                 1.000   1.000       13
gramophone          1.000   0.818       11
hedgehog            1.000   0.929       14
nautilus            1.000   0.933       15
pizza               1.000   1.000       13
pyramid             0.938   0.882       17
sea_horse           0.833   0.882       17
windsor_chair       0.941   1.000       16

=== densenet121 per-class precision / recall ===
                precision  recall  support
accordion           1.000   1.000       15
bass                1.000   1.000       14
camera              1.000   1.000       10
crocodile           0.778   0.700       10


## Observations
* All three ImageNet-pretrained backbones transfer well to these 15 Caltech categories
  even with the backbone **frozen** and only a linear head trained for 3 epochs, because
  the categories overlap strongly with ImageNet concepts.
* `densenet121` and `vgg19` typically edge out `resnet18` thanks to richer features,
  but all three reach high precision/recall on visually distinctive classes
  (e.g. `dollar_bill`, `pizza`, `accordion`).
* Classes that are visually similar (e.g. `crocodile` vs `crocodile_head`) show the
  lowest per-class precision/recall, which is the main source of confusion.